# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [2]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

scoper = config.chat().with_structured_output(Scope)

In [4]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

decision  20/24 right, 0.40s per call
chat      18/24 right, 9.40s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [ ]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)


  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


/Users/harshaprakash/agentic-ai-course/DADA2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.79     4.7         0.0   37.3
  diagnosis         0.86     2.4         2.1   20.8


In [7]:
for r in rows:
    print (r)

{'agent': 'ladder', 'accuracy': 0.7857142857142857, 'chat_calls': 4.714285714285714, 'decision_calls': 0.0, 'seconds': 37.27482865537916, 'n': 14}
{'agent': 'diagnosis', 'accuracy': 0.8571428571428571, 'chat_calls': 2.357142857142857, 'decision_calls': 2.142857142857143, 'seconds': 20.82499819142478, 'n': 14}


In [6]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

[ok ] What does MU state that the preparation of financial statements an  (2 chat, 2 decision)
        CORRECT
The provided answer accurately captures management's responsibility under U.S. GAAP as stated in MU's FY2024 filings.

[ok ] What termination fee is AMD required to pay ZT Systems under certa  (2 chat, 2 decision)
        CORRECT
The answer correctly identifies the $300 million termination fee associated with regulatory approvals as stated in the filings.

[ok ] What factors does NVIDIA believe are the principal competitive fac  (2 chat, 2 decision)
        CORRECT
The provided answer accurately lists the principal competitive factors for NVIDIA as stated in its fiscal year 2025 SEC filings.

[ok ] What competitive pressures from Intel and Nvidia did AMD face in f  (2 chat, 2 decision)
        CORRECT
The answer accurately summarizes the competitive pressures AMD faced from Intel and Nvidia in fiscal year 2025 as described in the provided co



If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

**Conditions.**
The corpus is 27 section files (Business, Risk Factors and MD&A) from nine 10-K filings by NVIDIA, AMD and Micron, plus their XBRL financials as JSON. The sections were split with RecursiveCharacterTextSplitter (chunk size 1200, overlap 150) into 2,124 chunks, and each chunk carries source, ticker, fiscal_year and section metadata. Chunks were embedded with @cf/qwen/qwen3-embedding-0.6b on Cloudflare Workers AI (1024 dimensions) and stored in a Pinecone serverless index with cosine similarity. Retrieval uses k = 5 (k = 10 on the comparative set), a rerank pool of 10 and an RRF constant of 60, and reranking uses Cohere rerank-v3.5. The chat model is gemini-3.5-flash-lite and the decision model is clef-flash on Cloudflare. The retrieval strategies in notebook 02 were developed on a six-question slice of the specific set, and on five comparative questions at k = 10. The routers were compared on all 24 labelled questions (14 specific, 10 comparative). The agents were compared on 14 questions (8 specific, 6 comparative), with answers judged by the chat model.

**The routers.**
On all 24 labelled questions, the decision model routed 20 correctly at 0.40 s per call, and the chat model routed 18 correctly at 9.40 s per call. The accuracy gap is two questions. The speed gap is larger than a model difference would explain, and it isn’t a fair measure of the chat model itself. chat_scope calls Gemini through retry, so any backoff from the free tier’s 15-requests-a-minute limit is counted as model time, while the decision model runs on Cloudflare with no such limit. Speed aside, the decision model earns its place for two reasons. First, it returns a probability for every option, and the agent acts on those numbers directly: route falls back to DEFAULT below SURE, and diagnose re-labels a low-confidence absent as wrong_words below ABSENT_SURE. In notebook 03, the TSMC search was diagnosed incomplete at only 0.38, while the dress-code search was absent at 0.90. A structured-output chat call would have returned only a label in both cases. Second, its calls don’t count against Gemini’s 500-a-day limit, which matters because the agents already spend 2.4 to 4.7 chat calls per question.

**The agents.**
On 14 questions, the diagnosis agent answered 0.86 correctly against 0.79 for the ladder, a difference of one question. It did this with about half the chat calls (2.4 per question against 4.7) and in less time (20.8 s per question against 37.3 s), plus 2.1 decision-model calls per question that don’t count against the Gemini limit. On the specific questions the judge sampled, the diagnosis agent took the shortest path in the graph every time: one filtered search, a sufficient verdict, and an answer, for 2 chat and 2 decision calls. The ladder can’t stop that early, because it has no verdict to stop on. The 4.2 run reports only averages, so it doesn’t show which questions the agents split on. The clearest difference I measured directly is from notebook 03, on the out-of-scope dress-code question. There, the ladder climbed to rung 2 and spent 9 chat calls over 3 attempts before saying the filings don’t mention it. The diagnosis agent stopped after a single search, because diagnose returned absent at 0.90, above ABSENT_SURE, and decide sends absent straight to answer. On the TSMC comparison, the ladder used 6 chat calls. The diagnosis agent’s first hybrid search came back incomplete, and because plan is told which sources it already has, it wrote only one new step for the missing company before the second diagnosis came back sufficient. The difference between the agents is what happens after a failure: the ladder retries the same way every time, while the diagnosis agent sends each verdict to a different node.

**What did not help.**
Multi-query RRF on top of the metadata filter. On the six-question slice in notebook 02, the filter alone reached recall@5 1.00 and MRR 0.917. Adding four rewrites fused with RRF kept recall at 1.00 but dropped MRR to 0.792. Adding the reranker only brought it back to 0.806, still below the filter alone, at roughly five extra calls per question. The reason is the order of the pipeline. Once the filter has narrowed the search to a single filing, there is almost nothing left for a different phrasing to find, and the rewrites can drift off-topic (one was “regulatory compliance hazards in global shipping”). RRF then rewards passages that the drifting rewrites agree on, which can push the single best passage down a rank. For questions that already name a company and a year, the filter does all the useful work.


**What I would ship.**
I would ship the diagnosis agent. It was slightly more accurate than the ladder (0.86 against 0.79 on 14 questions) and cost roughly half as much in chat calls and time. On specific questions I would drop multi-query fusion and use the plain filter, since it lowered MRR without improving recall. Before shipping, I would want to know what mix of questions users ask. The filter raised recall@5 from 0.67 to 1.00 on specific questions but dropped recall@10 from 0.83 to 0.57 on comparative ones, so the router’s thresholds should be tuned on real traffic rather than on 24 labelled questions. I would also want to know whether users expect the latest filing by default. When I asked the deployed app “Is anyone worried about China?”, it answered only from fiscal 2024 filings for all three companies, because similarity search has no notion of recency.

**Copy your write-up to README.md.**